In [ ]:
import os
import numpy as np

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PREFIX = os.path.join(PROJECT_ROOT, "data")

In [ ]:
def load_raw_data(prefix=DATA_PREFIX):
    """Carga los .npy de imágenes y máscaras (radiopedia + medseg)."""
    data = {
        "images_radiopedia": np.load(os.path.join(prefix, "images_radiopedia.npy")).astype(np.float32),
        "masks_radiopedia": np.load(os.path.join(prefix, "masks_radiopedia.npy")).astype(np.int8),
        "images_medseg": np.load(os.path.join(prefix, "images_medseg.npy")).astype(np.float32),
        "masks_medseg": np.load(os.path.join(prefix, "masks_medseg.npy")).astype(np.int8),
        "test_images_medseg": np.load(os.path.join(prefix, "test_images_medseg.npy")).astype(np.float32),
    }
    return data

In [ ]:
def check_integrity(images, masks, name=""):
    """Verifica que imágenes y máscaras tengan la misma cantidad y dimensiones espaciales."""
    assert len(images) == len(masks), f"{name}: cantidad de imágenes y máscaras no coincide ({len(images)} vs {len(masks)})"
    assert images.shape[1:3] == masks.shape[1:3], f"{name}: dimensiones espaciales no coinciden"
    print(f"{name}: OK ({len(images)} muestras, {images.shape[1:3]})")

In [ ]:
def preprocess_mask(mask: np.ndarray) -> np.ndarray:
    """Binariza la máscara a valores 0/1 (viene de dataset.ipynb, pertenece aquí)."""
    mask = mask.astype(np.float32)
    if mask.max() > 1.0:
        mask = mask / 255.0
    return mask

In [ ]:
def train_val_split(images, masks, val_ratio=0.2, seed=42):
    """Split simple train/val con shuffle reproducible."""
    n = len(images)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    n_val = int(n * val_ratio)

    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return (images[train_idx], masks[train_idx]), (images[val_idx], masks[val_idx])

In [ ]:
def image_statistics(images):
  mean = np.mean(images, axis=(0, 1, 2))
  std = np.std(images, axis=(0,1,2))
  max_pixel_value = 255.0
  return mean, std, max_pixel_value

: 

In [ ]:
def run_etl(prefix=DATA_PREFIX, val_ratio=0.2):
    """Orquesta el ETL completo: carga -> valida -> limpia máscaras -> split."""
    data = load_raw_data(prefix)

    check_integrity(data["images_radiopedia"], data["masks_radiopedia"], "radiopedia")
    check_integrity(data["images_medseg"], data["masks_medseg"], "medseg")

    data["masks_radiopedia"] = preprocess_mask(data["masks_radiopedia"])
    data["masks_medseg"] = preprocess_mask(data["masks_medseg"])

    (train_images, train_masks), (val_images, val_masks) = train_val_split(
        data["images_medseg"], data["masks_medseg"], val_ratio
    )

    return {
        "train_images": train_images, "train_masks": train_masks,
        "val_images": val_images, "val_masks": val_masks,
        "images_radiopedia": data["images_radiopedia"], "masks_radiopedia": data["masks_radiopedia"],
        "test_images_medseg": data["test_images_medseg"],
    }